# arc-witness-envs Quick Start

13 Witness-inspired puzzle environments for [ARC-AGI-3](https://arcprize.org/arc-agi/3/) — interactive reasoning challenges on a 64×64 pixel grid.

This notebook walks through:
1. Installation & setup
2. Visualizing a puzzle level
3. Step-by-step gameplay
4. Overview of all 13 games
5. Dataset statistics
6. Random agent baseline
7. Plugging in your own agent

## 1. Install & Setup

In [ ]:
# Install the ARC-AGI SDK
!pip install -q arc-agi

# Clone the repo (skip if running locally)
import os
if not os.path.exists("arc-witness-envs"):
    !git clone https://github.com/Guanghan/arc-witness-envs.git
os.chdir("arc-witness-envs")
print(f"Working directory: {os.getcwd()}")

In [ ]:
import sys, json, importlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from arcengine import GameAction, ActionInput, GameState

UP    = GameAction.ACTION1
DOWN  = GameAction.ACTION2
LEFT  = GameAction.ACTION3
RIGHT = GameAction.ACTION4
CONFIRM = GameAction.ACTION5
RESET = GameAction.RESET

ACTION_NAMES = {1: "UP", 2: "DOWN", 3: "LEFT", 4: "RIGHT", 5: "CONFIRM"}

# 16-color palette (index -> RGB)
PALETTE = {
    0:  [255, 255, 255],  # white
    1:  [204, 204, 204],  # light_gray
    2:  [153, 153, 153],  # gray
    3:  [102, 102, 102],  # dark_gray
    4:  [ 51,  51,  51],  # near_black
    5:  [  0,   0,   0],  # black
    6:  [229,  58, 163],  # magenta
    7:  [255, 123, 204],  # light_magenta
    8:  [249,  60,  49],  # red
    9:  [ 30, 147, 255],  # blue
    10: [136, 216, 241],  # light_blue
    11: [255, 220,   0],  # yellow
    12: [255, 133,  27],  # orange
    13: [146,  18,  49],  # maroon
    14: [ 79, 204,  48],  # green
    15: [163,  86, 214],  # purple
}

# Build matplotlib colormap from the palette
_colors_rgb = [np.array(PALETTE[i]) / 255.0 for i in range(16)]
WITNESS_CMAP = mcolors.ListedColormap(_colors_rgb)


def get_frame(game):
    """Get the current 64x64 grid from a game (as numpy array)."""
    frame_data = game.perform_action(ActionInput(id=RESET), raw=True)
    arr = frame_data.frame[0]
    return np.array(arr) if not isinstance(arr, np.ndarray) else arr


def act(game, action):
    """Perform an action and return (frame_array, frame_data)."""
    fd = game.perform_action(ActionInput(id=action), raw=True)
    arr = fd.frame[0]
    arr = np.array(arr) if not isinstance(arr, np.ndarray) else arr
    return arr, fd


def show_grid(grid, title="", ax=None):
    """Display a 64x64 color-index grid with the Witness palette."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(grid, cmap=WITNESS_CMAP, vmin=0, vmax=15,
              interpolation='nearest')
    ax.set_title(title, fontsize=10)
    ax.axis('off')


# Game registry: game_id -> (module_path, class_name)
GAME_REGISTRY = {
    f"tw{i:02d}": (f"environment_files.tw{i:02d}.tw{i:02d}", f"Tw{i:02d}")
    for i in range(1, 14)
}

def load_game(game_id, seed=0):
    """Load and return a game instance by ID."""
    mod_path, cls_name = GAME_REGISTRY[game_id]
    mod = importlib.import_module(mod_path)
    return getattr(mod, cls_name)(seed=seed)

print("Setup complete.")

## 2. Visualize a Puzzle Level

Each game renders to a **64×64 pixel grid** with 16 indexed colors. Let’s look at the first level of `tw01` (PathDots) — draw a path through all yellow waypoints.

In [ ]:
game = load_game("tw01")
grid = get_frame(game)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

# Left: the rendered grid
show_grid(grid, "tw01 PathDots — Level 1", ax=axes[0])

# Right: palette legend
semantic = {
    3: "Background", 5: "Grid lines", 9: "Path (blue)",
    11: "Cursor / Dots (yellow)", 14: "Start (green)", 8: "End (red)",
}
patches = []
labels = []
for idx, name in semantic.items():
    color = np.array(PALETTE[idx]) / 255.0
    patches.append(plt.Rectangle((0, 0), 1, 1, fc=color, ec='gray'))
    labels.append(f"{idx}: {name}")
axes[1].legend(patches, labels, loc='center', fontsize=10,
               title="Key Colors", title_fontsize=11, frameon=False)
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 3. Step-by-Step Gameplay

Solve `tw01` level 1 by executing actions one at a time. Watch the path grow on the grid.

In [ ]:
game = load_game("tw01")

# Known solution for tw01 level 1
solution = [RIGHT, RIGHT, UP, LEFT, LEFT, UP, RIGHT, RIGHT, CONFIRM]
action_labels = ["RIGHT", "RIGHT", "UP", "LEFT", "LEFT", "UP", "RIGHT", "RIGHT", "CONFIRM"]

# Collect frames at each step
frames = []
init_grid = get_frame(game)  # reset to get initial state
frames.append(("Start", init_grid.copy()))

for action, label in zip(solution, action_labels):
    grid, fd = act(game, action)
    frames.append((label, grid.copy()))

# Show key frames (every other + last two)
indices = [0, 2, 4, 6, 8, 9]  # Start, step 2, 4, 6, 8(RIGHT), 9(CONFIRM)
fig, axes = plt.subplots(1, len(indices), figsize=(3 * len(indices), 3))
for ax, i in zip(axes, indices):
    label, grid = frames[i]
    step_label = f"Step {i}: {label}" if i > 0 else "Initial"
    show_grid(grid, step_label, ax=ax)

plt.suptitle("tw01 PathDots — Solving Level 1", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Levels completed: {fd.levels_completed}")
print(f"Game state: {fd.state}")

## 4. All 13 Games at a Glance

Each game implements a different puzzle mechanic mapping to ARC-AGI [Core Knowledge](https://arxiv.org/abs/1911.01547) priors.

In [ ]:
game_titles = {
    "tw01": "PathDots",     "tw02": "ColorSplit",   "tw03": "ShapeFill",
    "tw04": "SymDraw",      "tw05": "StarPair",     "tw06": "TriCount",
    "tw07": "EraserLogic",  "tw08": "ComboBasic",   "tw09": "CylinderWrap",
    "tw10": "ColorFilter",  "tw11": "MultiRegion",  "tw12": "HexCombo",
    "tw13": "EraserAll",
}

fig, axes = plt.subplots(2, 7, figsize=(21, 6))
axes_flat = axes.flatten()

for i, (gid, title) in enumerate(game_titles.items()):
    game = load_game(gid)
    grid = get_frame(game)
    show_grid(grid, f"{gid}\n{title}", ax=axes_flat[i])

# Hide the extra subplot (14th cell in 2x7 grid)
axes_flat[13].axis('off')

plt.suptitle("All 13 Witness Games — Level 1", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 5. Dataset Statistics

Levels are stored in `levels/*.json`. Let’s visualize the distribution.

In [ ]:
stats = []
for gid in game_titles:
    path = f"levels/{gid}_levels.json"
    with open(path) as f:
        data = json.load(f)
    levels = data["levels"]
    validated = sum(1 for lv in levels if lv.get("validated", True))
    unvalidated = len(levels) - validated
    stats.append({"game": gid, "title": game_titles[gid],
                  "validated": validated, "unvalidated": unvalidated,
                  "total": len(levels)})

games = [s["game"] for s in stats]
val   = [s["validated"] for s in stats]
unval = [s["unvalidated"] for s in stats]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(games))
bar_w = 0.6
ax.bar(x, val, bar_w, label="Validated", color="#1e93ff")
ax.bar(x, unval, bar_w, bottom=val, label="Unvalidated", color="#ff851b", alpha=0.7)

# Annotate totals
for i, s in enumerate(stats):
    ax.text(i, s["total"] + 5, str(s["total"]), ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([f"{s['game']}\n{s['title']}" for s in stats],
                    rotation=45, ha='right', fontsize=8)
ax.set_ylabel("Number of Levels")
ax.set_title(f"Level Distribution Across 13 Games  (Total: {sum(s['total'] for s in stats):,})")
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f"Total: {sum(s['total'] for s in stats):,} levels "
      f"({sum(val)} validated + {sum(unval)} unvalidated)")

## 6. Random Agent Baseline

How hard are these puzzles? Let’s run a **random agent** that picks uniformly from the 5 actions. Spoiler: it almost never solves anything — showing that these puzzles require genuine reasoning.

In [ ]:
import random

ACTIONS = [UP, DOWN, LEFT, RIGHT, CONFIRM]

def random_agent_episode(game, max_steps=200):
    """Run one episode with random actions. Returns (solved, steps)."""
    fd = game.perform_action(ActionInput(id=RESET), raw=True)
    initial_completed = fd.levels_completed

    for step in range(1, max_steps + 1):
        action = random.choice(ACTIONS)
        fd = game.perform_action(ActionInput(id=action), raw=True)
        if fd.levels_completed > initial_completed:
            return True, step
    return False, max_steps

# Test on tw01 (simplest game) and tw02
results = {}
n_episodes = 100
max_steps = 200

for gid in ["tw01", "tw02", "tw06"]:
    game = load_game(gid)
    solves = 0
    total_steps = 0
    for _ in range(n_episodes):
        # Reset to level 1 each episode
        game = load_game(gid)
        solved, steps = random_agent_episode(game, max_steps)
        solves += solved
        total_steps += steps
    results[gid] = {
        "solve_rate": solves / n_episodes,
        "avg_steps": total_steps / n_episodes,
    }
    print(f"{gid} ({game_titles[gid]}): "
          f"solve rate = {solves}/{n_episodes} ({100*solves/n_episodes:.1f}%), "
          f"avg steps = {total_steps/n_episodes:.1f}")

print(f"\nRandom baseline is near 0% on most games — these puzzles require reasoning.")

## 7. Plug In Your Own Agent

Replace the `my_agent` function below with your own policy. The interface is simple:
- **Observation**: 64×64 int array (color indices 0–15)
- **Action**: one of 5 discrete actions (UP / DOWN / LEFT / RIGHT / CONFIRM)

In [ ]:
def my_agent(observation: np.ndarray) -> GameAction:
    """
    Your agent here.

    Args:
        observation: 64x64 numpy array of color indices (0-15)

    Returns:
        GameAction: one of ACTION1 (UP), ACTION2 (DOWN),
                    ACTION3 (LEFT), ACTION4 (RIGHT), ACTION5 (CONFIRM)
    """
    # TODO: replace with your policy
    return random.choice(ACTIONS)


# Run your agent on tw01
game = load_game("tw01")
fd = game.perform_action(ActionInput(id=RESET), raw=True)
initial_grid = np.array(fd.frame[0])

max_steps = 50
for step in range(max_steps):
    action = my_agent(initial_grid)
    grid, fd = act(game, action)
    initial_grid = grid

    if fd.levels_completed > 0:
        print(f"Level solved in {step + 1} steps!")
        show_grid(grid, f"Solved in {step+1} steps")
        plt.show()
        break
else:
    print(f"Not solved in {max_steps} steps. Keep improving your agent!")
    show_grid(grid, f"After {max_steps} steps (unsolved)")
    plt.show()

---

**Next steps:**
- Try the [OpenEnv adapter](https://github.com/Guanghan/arc-witness-envs#openenv-adapter-rl-training) for RL training (PPO, DQN, RND)
- Use `Arcade` from `arc-agi` SDK to [load these as ARC-AGI-3 environments](https://github.com/Guanghan/arc-witness-envs#arc-agi-3-integration)
- Play in the browser: `python play_human.py` → http://localhost:8001